# 06-state-machine.ipynb
## https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs-customer-support

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from langchain.chat_models import init_chat_model
model = init_chat_model('gpt-4.1-mini')

In [15]:
# Agent 내부의 Tool 활용에서 공통적으로 사용할 양식(State)를 생성

from langchain.agents import AgentState
# typing, pydantic, dataclass -> 라는 말들은 데이터의 형태를 설명하기 위함임

from typing_extensions import NotRequired
from typing import Literal

# SupportStep 은 말 그대로, [a, b, c] 중에 하나만 가능 -> 즉, 객관식 중 하나라고 생각하면 편할 듯(비워있어도 된다는거 참고)
SupportStep = Literal ['warranty_collector', 'issue_checker','solution_maker']

# AgentState 상속 받아서 시작
class SupportState(AgentState):
    """고객 지원 업무 흐름을 위한 State"""
    # Agent 내부에서 현재 어떤 단계를 진행중인지 확인용

    # 현재 단계
    current_step : NotRequired[SupportStep]   # NotRequired ( 비어 있어도 댐 -> 내용이 있다면 SupportStep literal 값 중 하나를 가짐)
    # 보증 상태
    warranty_status : NotRequired[Literal ['in_warranty', 'out_of_warranty']]  # LLM은 자연어를 제일 좋아함  <- Literal [True, False] 대신 자연어로 상태 설정
    # 이슈 타입
    issue_type : NotRequired[Literal['hw','sw']]


record_warranty_status  →  warranty_status 채움  +  current_step을 "issue_checker"로 변경
record_issue_type       →  issue_type 채움      +  current_step을 "solution_maker"로 변경
provide_solution        →  해결책 텍스트 리턴 (State 변경 없음, 최종 단계)
escalate_to_human       →  전문가 연결 텍스트 리턴 (State 변경 없음, 최종 단계)

In [16]:
# Tool 만들기 (중급)
# 기존 : Agent -> Tool  답변, 정보, 데이터 | 현재 : Tool -> Agent에게 명령

from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command  # langchain agent 가 langgraph 기반!

@tool
def record_warranty_status(
    # Agent가 판단해서 넣을 인자들
    status: Literal ['in_warranty', 'out_of_warranty'],
    # 이 도구가 일할 때 뭘 들고 오고(예시 -> API키, DB연결), 어디서 일하는지(조작할 상태)
    runtime : ToolRuntime[None, SupportState]
) -> Command:
    """사용자 보증상태를 기록하고, issu_checker로 넘김"""
    update_content = {
        'messages':[
            ToolMessage(
                content=f'보증 상태가 확인되었습니다.: {status}',
                tool_call_id=runtime.tool_call_id
            )
        ],
        'warranty_status': status,
        'current_step': 'issue_checker'  # 다음 Tool은 무엇인지 명시해줌 ( next_step으로 적어야 더 명확할듯,,)
    }
    
    # '명령'을 Agent에게 넘기는 구조임  -> AI 동작 제어!!  (Agent에게 시킬 일을 return)
    return Command(update=update_content)


@tool
def record_issue_type(
    issue_type: Literal["hw", "sw"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:  
    """이슈 타입을 기록하고, solution_maker에게 전달."""
    return Command(  
        update={  
            "messages": [
                ToolMessage(
                    content=f"이슈 종류는: {issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "solution_maker",  
        }
    )


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    # 실제 전문가에게 노티 보내기
    return f"전문가 지원으로 넘김. 사유: {reason}"


@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    # 실제로는 해결 방법을 검색해서 보내줌
    return f"해결 방안을 전달: {solution}"
 


                 Agent


     
보증 상태 Tool,                이슈 확인 Tool,  

끝나면
-> 메시지는 ~~
-> 보증 상태는 O,X
-> 현재 step: 이슈확인

위의 3가지가 Agent에 넘어감 

In [17]:
# 각 step 마다 필요한 설정 하기

#  기존 Agent  : 전체 작업 전용 SystemPrompt | 현재 Agent : 각 Step마다 다른 프롬프트

WARRANTY_COLLECTOR_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Warranty verification

At this step, you need to:
1. Greet the customer warmly
2. Ask if their device is under warranty
3. Use record_warranty_status to record their response and move to the next step

Be conversational and friendly. Don't ask multiple questions at once."""

ISSUE_CHECKER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Issue classification
CUSTOMER INFO: Warranty status is {warranty_status}

At this step, you need to:
1. Ask the customer to describe their issue
2. Determine if it's a hardware issue (physical damage, broken parts) or software issue (app crashes, performance)
3. Use record_issue_type to record the classification and move to the next step

If unclear, ask clarifying questions before classifying."""

SOLUTION_MAKER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SW issues: provide troubleshooting steps using provide_solution
2. For HW issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

Be specific and helpful in your solutions."""


# Step configuration: maps step name to (prompt, tools, required_state)
STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_checker": {
        "prompt": ISSUE_CHECKER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "solution_maker": {
        "prompt": SOLUTION_MAKER_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}



In [ ]:
# Step 기반의 미들웨어 만들기
# Middleware -> 작업 순서에서 특정 시점에 반드시 실행해야 하는 것 (사용자 질문 <-> 최종 답변 사이라고 생각하면 편함)

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def apply_step_config(
    # ModelRequest → LLM에게 보내는 질문/요청 (프롬프트, 메시지 등)
    # ModelResponse → LLM이 돌려주는 답변/응답 (생성된 텍스트, 도구 호출 등)
    # Callable -> ModelRequest를 넣으면 ModelResponse를 돌려주는 함수
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    # 현재 step 확인하기(시작-기본값은 warranty_collector 로 설정)
    # 1) State 칠판에서 현재 step 확인
    #    - 처음 시작할 때는 current_step이 없으므로 기본값 "warranty_collector"
    #    - 이후에는 Tool의 Command가 업데이트한 값이 들어있음
    current_step = request.state.get("current_step", "warranty_collector")  

    # 현재 step 에서 해야할 일을 확인
    # 2) STEP_CONFIG 딕셔너리에서 현재 step의 설정을 가져옴
    #    - prompt: 이 단계에서 LLM이 사용할 시스템 프롬프트
    #    - tools: 이 단계에서 LLM이 사용할 수 있는 도구 목록
    #    - requires: 이 단계에 오기 전에 반드시 채워져 있어야 할 State 필드들
    stage_config = STEP_CONFIG[current_step]  

    # 반드시 필요한 정보(보증상태, 이슈분류(HW, SW)) 검증
    # 3) 필수 State 값이 채워져 있는지 검증 (안전장치)
    #    - 예: issue_checker 단계에서는 warranty_status가 반드시 있어야 함
    #    - 빈 리스트면 (warranty_collector) 검증 없이 통과
    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    # Agent에 넣을 시스템 프롬프트를 생성 (프롬프트 에서 {변수명} 부분을 지금 채우는 과정)
    # 4) 시스템 프롬프트 생성
    #    - 프롬프트 템플릿의 {warranty_status}, {issue_type} 같은 변수를 State의 실제 값으로 채워넣음
    #    - 예: "Warranty status is {warranty_status}" → "Warranty status is in_warranty"
    system_prompt = stage_config["prompt"].format(**request.state)

    # 각 스텝마다, 시스템 프롬프트를 현재 상황(사용할 Tool)에 맞게 덮어 쓴다.
    # 5) request의 시스템 프롬프트 & 도구를 현재 step에 맞게 덮어쓰기
    #    - override()는 지정한 부분만 바꾸고, 사용자 메시지/대화 기록 등은 그대로 유지
    #    - 이렇게 하면 LLM은 현재 단계에 맞는 프롬프트를 받고, 해당 도구만 사용 가능
    request = request.override(  
        system_prompt=system_prompt,  
        tools=stage_config["tools"],  
    )

    return handler(request)


In [ ]:
# Agent 만들기 

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Collect all tools from all step configurations
all_tools = [
    record_warranty_status,
    record_issue_type,
    provide_solution,
    escalate_to_human,
]

agent = create_agent(
    model,
    tools=all_tools,
    # 내부적으로 state를 저장해야 한다는걸 명시
    # state = {'current_step': , 'warranty_status': , 'issue_type': } 빈 슬롯을 만드는 것과 같음
    state_schema=SupportState,  
    # 매 턴마다 LLM에게 요청 보내기 전에 먼저 실행됨
    # current_step 확인 → 해당 step의 시스템 프롬프트 & 사용 가능 도구를 세팅
    # 최종적으로 LLM에게 전달되는 것: 사용자 메시지 + 대화 기록 + 덮어쓴 시스템 프롬프트 + 덮어쓴 도구 목록
    middleware=[apply_step_config], 
    # 대화 턴마다 State를 메모리에 저장 → 다음 턴에서 이전 상태를 이어서 사용 가능
    # 없으면 매 턴마다 State가 초기화되어 warranty_status 같은 값이 날아감
    checkpointer=InMemorySaver(),  
)

In [20]:
# Test (스레드 id 바꾸고 싶을때만 실행)

import uuid

thread_id = uuid.uuid4()
config = {'configurable': {'thread_id': thread_id}}


In [ ]:
from langchain.messages import HumanMessage

print('--- 보증 확인 ---')
result = agent.invoke(
    {"messages":[{'role':'user','content':'나 핸드폰 이상해,,'}]}, config=config         
)



--- 보증 확인 ---


In [ ]:
from langchain.messages import HumanMessage

print('--- 시작 ---')
result = agent.invoke(
    {"messages":[{'role':'user','content':'나 핸드폰이 이상해,,'}]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}")

--- 시작 ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?


In [ ]:
from langchain.messages import HumanMessage

print('--- 보증 사실 관계 확인  ---')
result = agent.invoke(
    {"messages":[{'role':'user','content':'그게 뭐야,,,?'}]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}")

--- 보증 사실 관계 확인  ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

아, 보증 기간이라는 건 제품을 구매한 후 일정 기간 동안 제조사에서 무상으로 수리나 교환을 해주는 기간을 말해요. 혹시 구매한 지 얼마 안 됐거나, 구매할 때 받은 보증서나 영수증이 있으시면 그걸 기준으로 생각하시면 돼요. 사용 중인 핸드폰이 보증 기간 내에 있는지 확인해 주실 수 있을까요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

알려드린 내용이 조금 헷갈리셨나 봐요! 쉽게 말씀드리자면, 핸드폰을 산 지 얼마 안 된 상태인지, 아니면 산 지 꽤 오래돼서 무상 수리 기간이 지났는지를 묻는 거예요. 지금 핸드폰에 문제가 생겼는데, 혹시 아직 구매한 지 얼마 안 돼서 무상으로 도움을 받을 수 있는 상태인지 확인하고 싶어요. 핸드폰이 아직 보증 기간 내에 있다고 생각하세요? 네, 아니오로 편하게 말씀해 주세요!


In [24]:
from langchain.messages import HumanMessage

print('--- 보증 사실 관계 확인  ---')
result = agent.invoke(
    {"messages":[{'role':'user','content':'나 폰 산지 3년 정도 되었어'}]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}")

--- 보증 사실 관계 확인  ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

아, 보증 기간이라는 건 제품을 구매한 후 일정 기간 동안 제조사에서 무상으로 수리나 교환을 해주는 기간을 말해요. 혹시 구매한 지 얼마 안 됐거나, 구매할 때 받은 보증서나 영수증이 있으시면 그걸 기준으로 생각하시면 돼요. 사용 중인 핸드폰이 보증 기간 내에 있는지 확인해 주실 수 있을까요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

알려드린 내용이 조금 헷갈리셨나 봐요! 쉽게 말씀드리자면, 핸드폰을 산 지 얼마 안 된 상태인지, 아니면 산 지 꽤 오래돼서 무상 수리 기간이 지났는지를 묻는 거예요. 지금 핸드폰에 문제가 생겼는데, 혹시 아직 구매한 지 얼마 안 돼서 무상으로 도움을 받을 수 있는 상태인지 확인하고 싶어요. 핸드폰이 아직 보증 기간 내에 있다고 생각하세요? 네, 아니오로 편하게 말씀해 주세요!
=====

In [25]:
from langchain.messages import HumanMessage

print('--- 보증 사실 관계 확인  ---')
result = agent.invoke(
    {"messages":[{'role':'user','content':'나 보증이 남은거야?'}]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}")

--- 보증 사실 관계 확인  ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

아, 보증 기간이라는 건 제품을 구매한 후 일정 기간 동안 제조사에서 무상으로 수리나 교환을 해주는 기간을 말해요. 혹시 구매한 지 얼마 안 됐거나, 구매할 때 받은 보증서나 영수증이 있으시면 그걸 기준으로 생각하시면 돼요. 사용 중인 핸드폰이 보증 기간 내에 있는지 확인해 주실 수 있을까요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

알려드린 내용이 조금 헷갈리셨나 봐요! 쉽게 말씀드리자면, 핸드폰을 산 지 얼마 안 된 상태인지, 아니면 산 지 꽤 오래돼서 무상 수리 기간이 지났는지를 묻는 거예요. 지금 핸드폰에 문제가 생겼는데, 혹시 아직 구매한 지 얼마 안 돼서 무상으로 도움을 받을 수 있는 상태인지 확인하고 싶어요. 핸드폰이 아직 보증 기간 내에 있다고 생각하세요? 네, 아니오로 편하게 말씀해 주세요!
=====

In [26]:
from langchain.messages import HumanMessage

print('--- 문제 상황 확인  ---')
result = agent.invoke(
    {"messages":[HumanMessage('몰라 그냥 폰이 너무 느려')]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}")

--- 문제 상황 확인  ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

아, 보증 기간이라는 건 제품을 구매한 후 일정 기간 동안 제조사에서 무상으로 수리나 교환을 해주는 기간을 말해요. 혹시 구매한 지 얼마 안 됐거나, 구매할 때 받은 보증서나 영수증이 있으시면 그걸 기준으로 생각하시면 돼요. 사용 중인 핸드폰이 보증 기간 내에 있는지 확인해 주실 수 있을까요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

알려드린 내용이 조금 헷갈리셨나 봐요! 쉽게 말씀드리자면, 핸드폰을 산 지 얼마 안 된 상태인지, 아니면 산 지 꽤 오래돼서 무상 수리 기간이 지났는지를 묻는 거예요. 지금 핸드폰에 문제가 생겼는데, 혹시 아직 구매한 지 얼마 안 돼서 무상으로 도움을 받을 수 있는 상태인지 확인하고 싶어요. 핸드폰이 아직 보증 기간 내에 있다고 생각하세요? 네, 아니오로 편하게 말씀해 주세요!
========

In [ ]:
from langchain.messages import HumanMessage

print('--- 문제 상황 확인  ---')
result = agent.invoke(
    {"messages":[HumanMessage('다 해봤는데도 안돼')]}, config=config         
)

for msg in result['messages']:
    msg.pretty_print()

print(f"Current step: {result.get('current_step')}, warranty_status:{result.get('warranty_status')},  issue_type : {result.get('issue_type')}")

--- 문제 상황 확인  ---
================================ Human Message =================================

나 핸드폰이 이상해,,
================================== Ai Message ==================================

안녕하세요! 핸드폰이 이상하다고 하셨는데, 불편을 드려서 죄송해요. 혹시 사용하고 계신 핸드폰은 아직 보증 기간 내에 있나요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

아, 보증 기간이라는 건 제품을 구매한 후 일정 기간 동안 제조사에서 무상으로 수리나 교환을 해주는 기간을 말해요. 혹시 구매한 지 얼마 안 됐거나, 구매할 때 받은 보증서나 영수증이 있으시면 그걸 기준으로 생각하시면 돼요. 사용 중인 핸드폰이 보증 기간 내에 있는지 확인해 주실 수 있을까요?
================================ Human Message =================================

그게 뭐야,,,?
================================== Ai Message ==================================

알려드린 내용이 조금 헷갈리셨나 봐요! 쉽게 말씀드리자면, 핸드폰을 산 지 얼마 안 된 상태인지, 아니면 산 지 꽤 오래돼서 무상 수리 기간이 지났는지를 묻는 거예요. 지금 핸드폰에 문제가 생겼는데, 혹시 아직 구매한 지 얼마 안 돼서 무상으로 도움을 받을 수 있는 상태인지 확인하고 싶어요. 핸드폰이 아직 보증 기간 내에 있다고 생각하세요? 네, 아니오로 편하게 말씀해 주세요!
========